In [1]:
import h5py
import requests
import io
import fsspec
import numpy as np
from tqdm import tqdm  
import json
import tempfile
import os
from collections import Counter
from glob import glob

In [13]:
js_ai = sorted(glob(f"Lysine_*/AI_chrges*.json"))
js_resp = sorted(glob(f"Lysine_*/partial_charge*.json"))
print(js_ai, len(js_ai), js_resp, len(js_resp), sep ='\n')

['Lysine_1M/AI_chrges_Lysine_1M.json', 'Lysine_1M_ACC/AI_chrges_Lysine_1M_ACC.json', 'Lysine_2M/AI_chrges_Lysine_2M.json', 'Lysine_3M/AI_chrges_Lysine_3M.json', 'Lysine_ACC/AI_chrges_K_ACC.json', 'Lysine_Butyryl/AI_chrges_Lysine_butyryl.json', 'Lysine_Cro/AI_chrges_Kcr_H.json', 'Lysine_Formyl/AI_chrges_K_form.json', 'Lysine_Malonyl/AI_chrges_Lysine_malonyl.json', 'Lysine_lac/AI_chrges_Lysine_lac.json', 'Lysine_prop/AI_chrges_AKA_prop.json']
11
['Lysine_1M/partial_charges.json', 'Lysine_1M_ACC/partial_charges.json', 'Lysine_2M/partial_charges.json', 'Lysine_3M/partial_charges.json', 'Lysine_ACC/partial_charges.json', 'Lysine_Butyryl/partial_charges.json', 'Lysine_Butyryl/partial_charges_0.1278.json', 'Lysine_Cro/partial_charges.json', 'Lysine_Formyl/partial_charges.json', 'Lysine_Malonyl/partial_charges.json', 'Lysine_prop/partial_charges.json']
11


In [11]:
rtp_list = sorted(glob('amber14sb_mod.ff/Lysine_lac*.rtp'))
rtp_list

['amber14sb_mod.ff/Lysine_lac_ai.rtp']

In [12]:
import json

def replace_charge_in_line(line, new_charge):
    """Заменяет заряд в строке RTP с сохранением форматирования"""
    if not line.strip() or line.startswith(';'):
        return line
    
    # Извлекаем части по позициям
    atom_name = line[0:6].strip() if len(line) > 6 else ""
    atom_type = line[6:12].strip() if len(line) > 12 else ""
    idx = line[24:28].strip() if len(line) > 28 else ""
    
    # Формируем новую строку
    return f"{atom_name:>6}{atom_type:>6}{new_charge:12.4f}{idx:>4}\n"

In [13]:
for ptm in rtp_list[:]:
    ptm_type = ptm.split('/')[-1].split('.')[0]
    
    # Загружаем заряды
    with open(f'{ptm_type}/AI_chrges_{ptm_type}.json', 'r') as file:
        charge_ai = json.load(file)
    charge_ai = [round(x, 4) for x in charge_ai]
    
    # Создаём новый файл
    input_path = ptm
    output_path = f'amber14sb_mod.ff/{ptm_type}_ai.rtp'
    
    with open(input_path, 'r') as infile, open(output_path, 'w') as outfile:
        in_atoms_section = False
        atom_idx = 0
        
        for line in infile:
            # Определяем секции
            if '[ atoms ]' in line:
                in_atoms_section = True
                outfile.write(line)
                continue
            elif '[ bonds ]' in line:
                in_atoms_section = False
                outfile.write(line)
                continue
            
            # Заменяем заряды в секции atoms
            if in_atoms_section and line.strip() and not line.startswith(';'):
                if atom_idx < len(charge_ai):
                    new_line = replace_charge_in_line(line, charge_ai[atom_idx])
                    outfile.write(new_line)
                    atom_idx += 1
                else:
                    # Если зарядов больше нет, оставляем как есть
                    outfile.write(line)
            else:
                outfile.write(line)
    
    print(f"✅ Создан: {output_path}")
    print(f"   Заменено зарядов: {atom_idx}")

FileNotFoundError: [Errno 2] No such file or directory: 'Lysine_lac_ai/AI_chrges_Lysine_lac_ai.json'

In [14]:
for ptm in rtp_list:
    ptm_type = ptm.split('/')[-1].split('.')[0].replace('_ai','')
    
    # Загружаем заряды
    with open(f'{ptm_type}/partial_charges.json', 'r') as file:
        charge_ai = json.load(file)
    charge_ai = [round(x, 4) for x in charge_ai]
    
    # Создаём новый файл
    input_path = ptm
    output_path = f'amber14sb_mod.ff/{ptm_type}.rtp'
    
    with open(input_path, 'r') as infile, open(output_path, 'w') as outfile:
        in_atoms_section = False
        atom_idx = 0
        
        for line in infile:
            # Определяем секции
            if '[ atoms ]' in line:
                in_atoms_section = True
                outfile.write(line)
                continue
            elif '[ bonds ]' in line:
                in_atoms_section = False
                outfile.write(line)
                continue
            
            # Заменяем заряды в секции atoms
            if in_atoms_section and line.strip() and not line.startswith(';'):
                if atom_idx < len(charge_ai):
                    new_line = replace_charge_in_line(line, charge_ai[atom_idx])
                    outfile.write(new_line)
                    atom_idx += 1
                else:
                    # Если зарядов больше нет, оставляем как есть
                    outfile.write(line)
            else:
                outfile.write(line)
    
    print(f"✅ Создан: {output_path}")
    print(f"   Заменено зарядов: {atom_idx}")

✅ Создан: amber14sb_mod.ff/Lysine_lac.rtp
   Заменено зарядов: 30


## Создание rtp файла 

In [21]:
PTM_name

'Lysine_lac'

In [23]:
with open(f'{PTM_name}/AI_chrges_Lysine_lac.json', 'r') as charges:
    mod_acid_charges = json.load(charges)
print(f'{round(sum(mod_acid_charges), 4):.4}',mod_acid_charges, sep = '\n')

-0.0
[-0.34790000319480896, -0.23999999463558197, 0.7340999841690063, -0.5893999934196472, -0.19495677947998047, -0.20988492667675018, -0.2025669515132904, -0.003883455879986286, -0.6148805618286133, 0.3815005421638489, -0.4822674095630646, -0.04344647005200386, -0.7352320551872253, -0.358597069978714, 0.14259999990463257, 0.27469998598098755, 0.14171242713928223, 0.14171242713928223, 0.1396760791540146, 0.1396760642528534, 0.14945146441459656, 0.14945146441459656, 0.11691326647996902, 0.11691326647996902, 0.37381526827812195, 0.16579817235469818, 0.3953072428703308, 0.15322917699813843, 0.15322917699813843, 0.15322917699813843]


In [25]:
rtp = make_rtp_for_aminoacid(mod_base, charges=mod_acid_charges, name=base_name, shift=1, canonical_atoms = canonical_atoms)
# print(rtp)

with open(f'{PTM_name}/force_field_files/{PTM_name}.rtp' , 'w') as f: 
    f.write(rtp)
# with open(f'amber14sb_mod.ff/{PTM_name}.rtp' , 'w') as f: 
#     f.write(rtp)

In [26]:
print(rtp)

[ bondedtypes ]
; Col 1: Type of bond
; Col 2: Type of angles
; Col 3: Type of proper dihedrals
; Col 4: Type of improper dihedrals
; Col 5: Generate all dihedrals if 1, only heavy atoms of 0.
; Col 6: Number of excluded neighbors for nonbonded interactions
; Col 7: Generate 1,4 interactions between pairs of hydrogens if 1
; Col 8: Remove impropers over the same bond as a proper if it is 1
; bonds  angles  dihedrals  impropers all_dihedrals nrexcl HH14 RemoveDih
     1       1          9          4        1         3      1     0 

[ KLA ]
 [ atoms ]
     N     N     -0.3479   1
    CA    CX     -0.2400   2
     C     C      0.7341   3
     O     O     -0.5894   4
    CB    C8     -0.1950   5
    CG    C8     -0.2099   6
    CD    C8     -0.2026   7
    CE    C8     -0.0039   8
    NZ     N     -0.6149   9
    CH     C      0.3815  10
   OT1     O     -0.4823  11
   CT2    CT     -0.0434  12
   OI1    OH     -0.7352  13
   CI2    CT     -0.3586  14
     H     H      0.1426  15
    HA  